# This is to scan Top_Candidate_Pathogenic_repeats.csv files for the presence of repeats/genes in the Verifying_Keys.txt list

## 0.1 Load essential libraries to scan Top_Candidate_Pathogenic_repeats.csv files

In [1]:
import os
import glob
import pandas as pd
import openpyxl  # For .xlsx files
import json  # For .ipynb files (they're JSON format)
import csv

## 0.2 load the outpur folders and read the keys.txt
The Keys list has the following Columns LocusId	MotifID	CanonicalMotif	GencodeGeneName	GencodeGeneRegion	Pathogenic

In [2]:
# Define the folder pattern to search (change as needed)
folder_pattern = 'q05'  # e.g., 'q05', 'q01', etc.

# Base directory (where this notebook and Verifying_keys.txt are located)
base_dir = '/blue/zhou/leizhou/Agents/Biomni_Rpts_Ds/Rpt_Ds'

# Output directory (where the pattern-matched folders are located)
output_dir = '/blue/zhou/leizhou/Agents/Biomni_Rpts_Ds/Rpt_Ds/output'

# Find all folders matching the pattern (starting with the pattern)
matching_folders = [f for f in os.listdir(output_dir) 
                    if os.path.isdir(os.path.join(output_dir, f)) and f.startswith(folder_pattern)]

print(f"Base directory: {base_dir}")
print(f"Output directory: {output_dir}")
print(f"Folders matching '{folder_pattern}*': {matching_folders}")

# Load the Verifying_keys.txt file (in base_dir)
keys_file = os.path.join(base_dir, 'Verifying_keys.txt')

# Read the keys file as a DataFrame (tab-separated based on the column description)
keys_df = pd.read_csv(keys_file, sep='\t')

print(f"\nKeys file loaded: {keys_file}")
print(f"Number of keys: {len(keys_df)}")
print(f"Columns: {list(keys_df.columns)}")
print(f"\nFirst few rows:")
display(keys_df.head())

# Extract pathogenic keys for highlighting later
pathogenic_keys = keys_df[keys_df['Pathogenic'].str.upper() == 'YES']
print(f"\nNumber of pathogenic keys: {len(pathogenic_keys)}")

Base directory: /blue/zhou/leizhou/Agents/Biomni_Rpts_Ds/Rpt_Ds
Output directory: /blue/zhou/leizhou/Agents/Biomni_Rpts_Ds/Rpt_Ds/output
Folders matching 'q05*': ['q05_gem3p_Tmp0.5_200m_20251227-101203', 'q05_gpt5_Tmp0.9_200m_20260102-193716', 'q05_gpt4o_Tmp0.7_200m_20251231-051750', 'q05_grk4_Tmp0.9_200m_20251225-143615', 'q05_s4.5_Tmp0.7_200m_20251223-145132', 'q05_gem3p_Tmp0.7_200m_20251227-102027', 'q05_gpt52_Tmp0.9_200m_20251226-045202', 'q05_o4.5_Tmp0.9_200m_20251220-182331', 'q05_grk4_Tmp0.7_200m_20251225-140927', 'q05_s4.5_Tmp0.5_200m_20251223-105747', 'q05_o4.5_Tmp0.7_200m_20251220-172453', 'q05_gpt4o_Tmp0.9_200m_20251231-055730', 'q05_gpt4o_Tmp0.5_200m_20251231-050314', 'q05_s3.5_Tmp0.7_200m_20251231-050257', 'q05_s4.5_Tmp0.9_200m_20251223-190444', 'q05_gpt5_Tmp0.7_200m_20260102-141054', 'q05_grk4_Tmp0.5_200m_20251225-133248', 'q05_s3.5_Tmp0.5_200m_20251231-050255', 'q05_gpt52_Tmp0.7_200m_20251226-042101', 'q05_gpt5_Tmp0.5_200m_20260102-122433', 'q05_s3.5_Tmp0.9_200m_20251231

,LocusId,MotifID,CanonicalMotif,GencodeGeneName,GencodeGeneRegion,Pathogenic
0,X-148500631-148500691-GCC,GCC,CCG,AFF2,5' UTR,YES
1,2-190880872-190880920-GCA,GCA,AGC,GLS,5' UTR,YES
2,1-146228800-146228821-GCC,GCC,CCG,NOTCH2NLA,5' UTR,Possible
3,10-93702522-93702546-CCG,CCG,CCG,FRA10AC1,5' UTR,Possible
4,11-66744819-66744843-GGC,GGC,CCG,TOP6BL,5' UTR,Possible



Number of pathogenic keys: 2


## 1.1 Parse Top_Candidate_Pathogenic_repeats.csv in each targeted folder and report occurrence of Keys. Show matched keys with their row position (e.g., GLS(2), AFF2(5)). Highlight the occurrence of Keys with Pathogenic=YES

In [3]:
def read_csv_as_dataframe(filepath):
    """Read CSV file and return as DataFrame."""
    try:
        df = pd.read_csv(filepath)
        return df
    except Exception as e:
        print(f"  Warning: Could not read {filepath}: {e}")
        return None

def find_key_in_csv_with_position(df, key):
    """Find key in CSV DataFrame and return the row positions (1-indexed)."""
    if df is None:
        return []
    
    positions = []
    # Convert DataFrame to string representation for each cell and search
    for row_idx in range(len(df)):
        for col in df.columns:
            cell_value = str(df.iloc[row_idx][col]).lower()
            if key.lower() in cell_value:
                # Row position is 1-indexed (row 1 = first data row after header)
                positions.append(row_idx + 1)
                break  # Only count once per row
    return positions

# Build key pairs from each row (LocusId + GencodeGeneName treated as one unit)
# If both are found in a file, count as ONE match, not two
key_pairs = []
for idx, row in keys_df.iterrows():
    pathogenic = str(row['Pathogenic']).strip().upper()
    locus_id = str(row['LocusId']).strip() if pd.notna(row['LocusId']) else None
    gene_name = str(row['GencodeGeneName']).strip() if pd.notna(row['GencodeGeneName']) else None
    
    # Skip if both are invalid
    if (locus_id is None or locus_id.lower() == 'nan') and (gene_name is None or gene_name.lower() == 'nan'):
        continue
    
    # Clean up values
    if locus_id and locus_id.lower() == 'nan':
        locus_id = None
    if gene_name and gene_name.lower() == 'nan':
        gene_name = None
    
    key_pairs.append({
        'locus_id': locus_id,
        'gene_name': gene_name,
        'pathogenic': pathogenic,
        'row_idx': idx
    })

print(f"Total key pairs (LocusId + GeneName as one unit): {len(key_pairs)}")
print(f"YES pairs: {sum(1 for p in key_pairs if p['pathogenic'] == 'YES')}")
print(f"POSSIBLE pairs: {sum(1 for p in key_pairs if p['pathogenic'] == 'POSSIBLE')}")

# Target file name to scan
TARGET_FILE = "Top_Candidate_Pathogenic_repeats.csv"

# Scan only Top_Candidate_Pathogenic_repeats.csv in matching folders
file_results = []
no_match_files = []  # Track files with no matches
missing_target_files = []  # Track folders missing the target file

for folder in matching_folders:
    folder_path = os.path.join(output_dir, folder)
    target_filepath = os.path.join(folder_path, TARGET_FILE)
    
    # Check if target file exists
    if not os.path.exists(target_filepath):
        missing_target_files.append({
            'folder': folder,
            'expected_file': TARGET_FILE
        })
        print(f"  Warning: {TARGET_FILE} not found in {folder}")
        continue
    
    print(f"\nScanning: {target_filepath}")
    
    # Read CSV as DataFrame
    df = read_csv_as_dataframe(target_filepath)
    if df is None:
        continue
    
    # Also get string content for context display
    content = df.to_string()
    
    file_score = 0
    file_matches = []
    yes_matches = []
    matched_keys_with_positions = []  # New: track keys with their positions
    
    # Check each key pair - count as ONE match if either or both keys found
    for pair in key_pairs:
        locus_positions = []
        gene_positions = []
        
        if pair['locus_id']:
            locus_positions = find_key_in_csv_with_position(df, pair['locus_id'])
        if pair['gene_name']:
            gene_positions = find_key_in_csv_with_position(df, pair['gene_name'])
        
        # If either key found, count as ONE match for this pair
        if locus_positions or gene_positions:
            pathogenic = pair['pathogenic']
            
            # Determine which keys were found and at what positions
            found_keys = []
            key_position_strs = []
            
            if locus_positions:
                found_keys.append(f"LocusId:{pair['locus_id']}")
                # Add to position strings
                for pos in locus_positions:
                    key_position_strs.append(f"{pair['locus_id']}({pos})")
            if gene_positions:
                found_keys.append(f"GeneName:{pair['gene_name']}")
                # Add to position strings (gene names are usually more meaningful)
                for pos in gene_positions:
                    key_position_strs.append(f"{pair['gene_name']}({pos})")
            
            # Score based on pathogenic status (1 match per pair regardless of both keys found)
            if pathogenic == 'YES':
                points = 50
            elif pathogenic == 'POSSIBLE':
                points = 10
            else:
                points = 1
            
            file_score += points
            
            # All positions found (union of locus and gene positions)
            all_positions = sorted(set(locus_positions + gene_positions))
            
            file_matches.append({
                'key_pair': f"{pair['locus_id'] or 'N/A'} / {pair['gene_name'] or 'N/A'}",
                'found_keys': found_keys,
                'pathogenic': pathogenic,
                'count': 1,  # Always 1 per pair
                'points': points,
                'both_found': bool(locus_positions and gene_positions),
                'positions': all_positions,
                'position_strs': key_position_strs
            })
            
            # Add to matched keys with positions list
            matched_keys_with_positions.extend(key_position_strs)
            
            if pathogenic == 'YES':
                yes_matches.append({
                    'key_pair': f"{pair['locus_id'] or 'N/A'} / {pair['gene_name'] or 'N/A'}",
                    'found_keys': found_keys,
                    'both_found': bool(locus_positions and gene_positions),
                    'positions': all_positions,
                    'position_strs': key_position_strs
                })
    
    relative_path = os.path.relpath(target_filepath, output_dir)
    
    if file_score > 0:
        file_results.append({
            'file': relative_path,
            'folder': folder,
            'total_points': file_score,
            'matches': file_matches,
            'yes_matches': yes_matches,
            'yes_count': len(yes_matches),
            'possible_count': sum(1 for m in file_matches if m['pathogenic'] == 'POSSIBLE'),
            'matched_keys_positions': ', '.join(matched_keys_with_positions)  # New: string of all matched keys with positions
        })
    else:
        # Track files with no matches
        no_match_files.append({
            'file': relative_path,
            'folder': folder
        })

# Sort by total points (highest first)
file_results = sorted(file_results, key=lambda x: x['total_points'], reverse=True)

print(f"\n{'='*80}")
print(f"TARGET FILE: {TARGET_FILE}")
print(f"RESULTS: Found {len(file_results)} files with key matches")
print(f"         Found {len(no_match_files)} files with NO matches")
print(f"         Found {len(missing_target_files)} folders missing the target file")
print(f"{'='*80}")

Total key pairs (LocusId + GeneName as one unit): 10
YES pairs: 2
POSSIBLE pairs: 8

Scanning: /blue/zhou/leizhou/Agents/Biomni_Rpts_Ds/Rpt_Ds/output/q05_gpt4o_Tmp0.7_200m_20251231-051750/Top_Candidate_Pathogenic_repeats.csv



Scanning: /blue/zhou/leizhou/Agents/Biomni_Rpts_Ds/Rpt_Ds/output/q05_s4.5_Tmp0.7_200m_20251223-145132/Top_Candidate_Pathogenic_repeats.csv

Scanning: /blue/zhou/leizhou/Agents/Biomni_Rpts_Ds/Rpt_Ds/output/q05_gpt52_Tmp0.9_200m_20251226-045202/Top_Candidate_Pathogenic_repeats.csv

Scanning: /blue/zhou/leizhou/Agents/Biomni_Rpts_Ds/Rpt_Ds/output/q05_o4.5_Tmp0.9_200m_20251220-182331/Top_Candidate_Pathogenic_repeats.csv

Scanning: /blue/zhou/leizhou/Agents/Biomni_Rpts_Ds/Rpt_Ds/output/q05_grk4_Tmp0.7_200m_20251225-140927/Top_Candidate_Pathogenic_repeats.csv

Scanning: /blue/zhou/leizhou/Agents/Biomni_Rpts_Ds/Rpt_Ds/output/q05_s4.5_Tmp0.5_200m_20251223-105747/Top_Candidate_Pathogenic_repeats.csv

Scanning: /blue/zhou/leizhou/Agents/Biomni_Rpts_Ds/Rpt_Ds/output/q05_o4.5_Tmp0.7_200m_20251220-172453/Top_Candidate_Pathogenic_repeats.csv

Scanning: /blue/zhou/leizhou/Agents/Biomni_Rpts_Ds/Rpt_Ds/output/q05_gpt4o_Tmp0.5_200m_20251231-050314/Top_Candidate_Pathogenic_repeats.csv

Scanning: /blue/z

In [ ]:
# LLM short ID to full name mapping
LLM_NAME_MAP = {
    'o4.5': 'Opus 4.5',
    's4.5': 'Sonnet 4.5',
    'gem3p': 'Gemini 3 Pro',
    'gem3f': 'Gemini 3 Flash',
    'gpt52': 'GPT 5.2',
    'grk4': 'Grok 4',
    'gpt4o': 'GPT 4o',
    'gem2': 'Gemini 2.0'
}

def parse_folder_name(folder_name):
    """Parse folder name to extract LLM and Temperature.
    Expected format: q05_gem3p_Tmp0.5_200m_20251225-160005
    """
    parts = folder_name.split('_')
    llm_short = parts[1] if len(parts) > 1 else 'unknown'
    llm_full = LLM_NAME_MAP.get(llm_short, llm_short)
    
    # Extract temperature from Tmp part
    temp = 'N/A'
    for part in parts:
        if part.startswith('Tmp'):
            temp = part.replace('Tmp', '')
            break
    
    return llm_full, temp

def get_gene_only_positions(matches):
    """Extract only gene name positions (not LocusId) from matches."""
    gene_positions = []
    for match in matches:
        # Look for gene_name in the match
        for key in match.get('found_keys', []):
            if key.startswith('GeneName:'):
                gene_name = key.replace('GeneName:', '')
                # Get positions for this gene
                for pos_str in match.get('position_strs', []):
                    # Check if this position string is for the gene (not locus)
                    if gene_name in pos_str:
                        gene_positions.append(pos_str)
    return ', '.join(gene_positions) if gene_positions else 'N/A'

# Display ranked results
for i, result in enumerate(file_results, 1):
    print(f"\n{'─'*80}")
    print(f"RANK #{i}: {result['folder']}")
    print(f"  File: {result['file']}")
    print(f"  Total Points: {result['total_points']} | YES matches: {result['yes_count']} | POSSIBLE matches: {result['possible_count']}")
    print(f"  Matched Keys (position): {result['matched_keys_positions']}")
    
    # Show Pathogenic=YES match details
    if result['yes_matches']:
        print(f"\n  🔴 PATHOGENIC=YES MATCHES:")
        for ym in result['yes_matches']:
            both_indicator = "✓ BOTH" if ym['both_found'] else "partial"
            print(f"    Key Pair: '{ym['key_pair']}' [{both_indicator}]")
            print(f"    Found at positions: {ym['position_strs']}")
            print()

# Create a summary DataFrame for files with matches
summary_data = []
for i, r in enumerate(file_results, 1):
    llm_name, temp = parse_folder_name(r['folder'])
    gene_only_positions = get_gene_only_positions(r['matches'])
    
    summary_data.append({
        'LLM': llm_name,
        'Temperature': temp,
        'Total_Points': r['total_points'],
        'YES_Matches': r['yes_count'],
        'POSSIBLE_Matches': r['possible_count'],
        'Matched_Keys_Position': gene_only_positions
    })

summary_df = pd.DataFrame(summary_data)

print(f"\n{'='*80}")
print("SUMMARY TABLE - FILES WITH MATCHES")
print(f"{'='*80}")
display(summary_df)

# Create a summary DataFrame for files with NO matches
if no_match_files:
    no_match_df = pd.DataFrame(no_match_files)
    print(f"\n{'='*80}")
    print(f"FILES WITH NO MATCHES ({len(no_match_files)} files)")
    print(f"{'='*80}")
    display(no_match_df)
else:
    no_match_df = pd.DataFrame(columns=['file', 'folder'])
    print(f"\n✅ All {TARGET_FILE} files had at least one match!")

# Show missing target files
if missing_target_files:
    missing_df = pd.DataFrame(missing_target_files)
    print(f"\n{'='*80}")
    print(f"FOLDERS MISSING {TARGET_FILE} ({len(missing_target_files)} folders)")
    print(f"{'='*80}")
    display(missing_df)

# Save summary as markdown table and CSV files
from datetime import datetime
date_str = datetime.now().strftime('%Y%m%d')
md_filename = f"verify_{folder_pattern}_{date_str}.md"
md_filepath = os.path.join(base_dir, md_filename)

# Save CSV files
csv_summary_filepath = os.path.join(base_dir, f"verify_{folder_pattern}_{date_str}_summary.csv")
csv_no_match_filepath = os.path.join(base_dir, f"verify_{folder_pattern}_{date_str}_no_matches.csv")
csv_missing_filepath = os.path.join(base_dir, f"verify_{folder_pattern}_{date_str}_missing_files.csv")

# Save summary DataFrame to CSV
if len(summary_df) > 0:
    summary_df.to_csv(csv_summary_filepath, index=False)
    print(f"✅ Summary CSV saved to: {csv_summary_filepath}")

# Save no matches DataFrame to CSV
if len(no_match_files) > 0:
    no_match_df.to_csv(csv_no_match_filepath, index=False)
    print(f"✅ No matches CSV saved to: {csv_no_match_filepath}")

# Save missing files DataFrame to CSV
if len(missing_target_files) > 0:
    pd.DataFrame(missing_target_files).to_csv(csv_missing_filepath, index=False)
    print(f"✅ Missing files CSV saved to: {csv_missing_filepath}")

with open(md_filepath, 'w') as f:
    f.write(f"# Verification Results for {folder_pattern}\n\n")
    f.write(f"**Date:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write(f"**Output Directory:** {output_dir}\n\n")
    f.write(f"**Target File:** {TARGET_FILE}\n\n")
    f.write(f"**Folders Scanned:** {matching_folders}\n\n")
    f.write("## Summary Table - Files with Matches\n\n")
    if len(summary_df) > 0:
        f.write(summary_df.to_markdown(index=False))
    else:
        f.write("*No files with matches found.*\n")
    f.write("\n\n")
    
    # Add section for files with no matches
    f.write("## Files with No Matches\n\n")
    if len(no_match_files) > 0:
        f.write(f"**Total files with no matches:** {len(no_match_files)}\n\n")
        f.write(no_match_df.to_markdown(index=False))
    else:
        f.write(f"*All {TARGET_FILE} files had at least one match.*\n")
    f.write("\n\n")
    
    # Add section for missing target files
    f.write(f"## Folders Missing {TARGET_FILE}\n\n")
    if len(missing_target_files) > 0:
        f.write(f"**Total folders missing target file:** {len(missing_target_files)}\n\n")
        f.write(pd.DataFrame(missing_target_files).to_markdown(index=False))
    else:
        f.write(f"*All folders contain {TARGET_FILE}.*\n")
    f.write("\n")

print(f"✅ Markdown saved to: {md_filepath}")


────────────────────────────────────────────────────────────────────────────────
RANK #1: q05_o4.5_Tmp0.7_200m_20251220-172453
  File: q05_o4.5_Tmp0.7_200m_20251220-172453/Top_Candidate_Pathogenic_repeats.csv
  Total Points: 160 | YES matches: 2 | POSSIBLE matches: 6
  Matched Keys (position): X-148500631-148500691-GCC(15), AFF2(15), 2-190880872-190880920-GCA(5), GLS(5), 10-93702522-93702546-CCG(31), FRA10AC1(31), 11-119206289-119206322-CGG(33), CBL(33), CSNK1E(47), 7-55887600-55887639-GCG(12), ENSG00000249773(12), X-19990922-19990973-CCG(9), BCLAF3(9), X-149631735-149631780-CGC(4), TMEM185A(4)

  🔴 PATHOGENIC=YES MATCHES:
    Key Pair: 'X-148500631-148500691-GCC / AFF2' [✓ BOTH]
    Found at positions: ['X-148500631-148500691-GCC(15)', 'AFF2(15)']

    Key Pair: '2-190880872-190880920-GCA / GLS' [✓ BOTH]
    Found at positions: ['2-190880872-190880920-GCA(5)', 'GLS(5)']


────────────────────────────────────────────────────────────────────────────────
RANK #2: q05_gpt5_Tmp0.5_200m_20

,LLM,Temperature,Total_Points,YES_Matches,POSSIBLE_Matches,Matched_Keys_Position
0,Opus 4.5,0.7,160,2,6,"AFF2(15), GLS(5), FRA10AC1(31), CBL(33), CSNK1..."
1,gpt5,0.5,160,2,6,"AFF2(11), AFF2(12), AFF2(15), GLS(6), FRA10AC1..."
2,Opus 4.5,0.9,150,2,5,"AFF2(11), GLS(4), FRA10AC1(28), CBL(17), ENSG0..."
3,GPT 5.2,0.9,140,2,4,"AFF2(6), GLS(1), CBL(18), ENSG00000249773(15),..."
4,Opus 4.5,0.5,140,2,4,"AFF2(1), GLS(3), NOTCH2NLA(45), CBL(39), BCLAF..."
5,Sonnet 4.5,0.5,100,1,5,"GLS(1), FRA10AC1(5), CBL(3), ENSG00000249773(4..."
6,GPT 5.2,0.7,90,1,4,"GLS(2), CBL(16), ENSG00000249773(6), BCLAF3(20..."
7,GPT 5.2,0.5,80,1,3,"GLS(3), CBL(2), CSNK1E(50), TMEM185A(1)"
8,Sonnet 4.5,0.7,40,0,4,"CBL(13), CSNK1E(17), CSNK1E(18), BCLAF3(25), T..."
9,Gemini 3 Pro,0.9,10,0,1,CBL(12)



FILES WITH NO MATCHES (4 files)


,file,folder
0,q05_gpt4o_Tmp0.7_200m_20251231-051750/Top_Cand...,q05_gpt4o_Tmp0.7_200m_20251231-051750
1,q05_grk4_Tmp0.7_200m_20251225-140927/Top_Candi...,q05_grk4_Tmp0.7_200m_20251225-140927
2,q05_gpt4o_Tmp0.5_200m_20251231-050314/Top_Cand...,q05_gpt4o_Tmp0.5_200m_20251231-050314
3,q05_gpt5_Tmp0.7_200m_20260102-141054/Top_Candi...,q05_gpt5_Tmp0.7_200m_20260102-141054



FOLDERS MISSING Top_Candidate_Pathogenic_repeats.csv (10 folders)


,folder,expected_file
0,q05_gem3p_Tmp0.5_200m_20251227-101203,Top_Candidate_Pathogenic_repeats.csv
1,q05_gpt5_Tmp0.9_200m_20260102-193716,Top_Candidate_Pathogenic_repeats.csv
2,q05_grk4_Tmp0.9_200m_20251225-143615,Top_Candidate_Pathogenic_repeats.csv
3,q05_gem3p_Tmp0.7_200m_20251227-102027,Top_Candidate_Pathogenic_repeats.csv
4,q05_gpt4o_Tmp0.9_200m_20251231-055730,Top_Candidate_Pathogenic_repeats.csv
5,q05_s3.5_Tmp0.7_200m_20251231-050257,Top_Candidate_Pathogenic_repeats.csv
6,q05_s4.5_Tmp0.9_200m_20251223-190444,Top_Candidate_Pathogenic_repeats.csv
7,q05_grk4_Tmp0.5_200m_20251225-133248,Top_Candidate_Pathogenic_repeats.csv
8,q05_s3.5_Tmp0.5_200m_20251231-050255,Top_Candidate_Pathogenic_repeats.csv
9,q05_s3.5_Tmp0.9_200m_20251231-050300,Top_Candidate_Pathogenic_repeats.csv



✅ Summary saved to: /blue/zhou/leizhou/Agents/Biomni_Rpts_Ds/Rpt_Ds/verify_q05_20260113.md
